# GW assignment 3

In [ ]:
import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import astropy.units as u
import astropy.cosmology as cosmo
from astropy.cosmology import FlatwCDM
cosmol = FlatwCDM(H0=67.9, Om0=0.3065, w0=-1)

import numpy as np
import matplotlib.pyplot as plt

from gwpy.timeseries import TimeSeries as GWPYTimeSeries
from pycbc.filter import resample_to_delta_t, highpass
from pycbc.psd import interpolate, inverse_spectrum_truncation, aLIGOZeroDetHighPower
from pycbc.types import TimeSeries, FrequencySeries
from pycbc.filter import matched_filter
from pycbc.noise import noise_from_psd
from pycbc.detector import Detector
from pycbc.waveform import get_td_waveform
import bilby

### Q1a: Generate the GW waveform of a merger between two 20 solar mass black holes at a luminosity distance of 1000 Mpc. Inject the GW signal into Gaussian noise to obtain the strain. Plot the strain, with the injected waveform overlaid. (hint: see workshop tutorial 2.1 for help aligning the waveform properly)

In [ ]:
#you should use these parameters for your waveform and noise
sample_rate = 2048 # samples per second
duration = 32 # seconds
f_lower = 15.0 # lowest frequency in Hz

delta_t = 1.0 / sample_rate
approximant = "IMRPhenomD"

#this is the PSD you should use for the noise
psd = aLIGOZeroDetHighPower(sample_rate, duration, f_lower)
#Your code here



### Q1b: plot the Q transform of the strain, such that the merger is visible. Plot the transform with different q ranges. What is the effect of changing the the qrange parameter on the Q transform?

In [ ]:
#Your code here


### Q1c: Pick a gravitational wave signal from the GWTC3 catalogue: https://link.aps.org/doi/10.1103/PhysRevX.13.041039 (see table IV, or the [GWOSC webpage](https://gwosc.org/eventapi/html/query/show?release=GWTC-3-confident)) with an SNR above 15 and fetch the strain using gwpy. Plot its Q transform. (note the function to fetch the data is aliased to GWPYTimeSeries.fetch_open_data to avoid confusion with PyCBC's TimeSeries )

In [ ]:
from pycbc.catalog import Merger

#Your code here


### Q1d: Use gwpy to download the Hanford and Livingston data 8 seconds either side of the gps time given below. Plot the Q transform of the data for each detector. What do you observe in the Q transform? Is there a visible gravitational wave signal?

In [ ]:
duration = 16
gps_time = 1244875090.25

#Your code here

### Q2: Use matched filtering to plot the SNR time series for the signal you generated in Q1a. What is the SNR of the signal?

In [ ]:
#Your code here

In [ ]:
#need to crop to get rid of filter ringing on the edges
crop = 1
snr.data[:crop*sample_rate] = 0
snr.data[-crop*sample_rate:] = 0

#taking the absolute value of the snr as we don't care about phase
snr = np.abs(snr)

times = np.linspace(0, duration, len(snr))
plt.plot(times, snr)

plt.ylabel("SNR")
plt.xlabel("time (s)")

### Q3. Use matched filtering to identify multiple signals in a section of pregenerated noise. For each signal, find the time of the merger, and the approximate chirp mass of the signal. Plot the SNR time series for one of the detectors.

There are 5 signals to find, and all masses are a multiple of 5 and are in the range 5 - 50 solar masses. None of the signals are injected in the first or last 100 seconds of the data. Hint: The start of the SNR time series will be corrupted due to filter wraparound. This will be proportional to the length of the filter.

In [ ]:
import numpy as np

In [ ]:
start_time = 1240210479
sample_rate = 4096
duration = 1024
f_lower = 30

delta_t = 1.0 / sample_rate
delta_f = 1.0 /duration
ifos = ['H1', 'L1']

strain = np.load("Q3_strain_data.npz")
psds = np.load("Q3_psd_data.npz")

strain = {ifo: TimeSeries(strain[ifo], delta_t=delta_t) for ifo in ifos}
psds = {ifo: FrequencySeries(psds[ifo], delta_f=delta_f) for ifo in ifos}

### Q3b: In Q3a, you were searching for the detector-frame masses. Pick one of the signals from the previous part, and use Bilby to estimate the signal's chirp mass, the source frame masses, and luminosity distance of the signal. You may assume spin, sky location, inclination etc are all zero.

In [ ]:
#add the merger time, in seconds, of the signal you want to analyse
mtime = 

In [ ]:

H1 = bilby.gw.detector.get_empty_interferometer("H1")
L1 = bilby.gw.detector.get_empty_interferometer("L1")

pre_trigger = 2
post_trigger = 2
H1.set_strain_data_from_gwpy_timeseries(GWPYTimeSeries(strain["H1"][(mtime-pre_trigger)* sample_rate:(mtime+post_trigger)*sample_rate], sample_rate=sample_rate, t0=start_time+ (mtime-pre_trigger)))
L1.set_strain_data_from_gwpy_timeseries(GWPYTimeSeries(strain["L1"][(mtime-pre_trigger)* sample_rate:(mtime+post_trigger)*sample_rate], sample_rate=sample_rate, t0=start_time+ (mtime-pre_trigger)))
#L1.set_strain_data_from_gwpy_timeseries(noise_gwpy["L1"][(mtime-2)* sample_rate:(mtime+2)*sample_rate])

duration = pre_trigger + post_trigger
psd_duration = 128
H1_psd_data = GWPYTimeSeries(strain["H1"][(mtime - 2 - psd_duration)* sample_rate:(mtime - 2)*sample_rate], sample_rate=sample_rate)
L1_psd_data = GWPYTimeSeries(strain["L1"][(mtime - 2 - psd_duration)* sample_rate:(mtime - 2)*sample_rate], sample_rate=sample_rate)

psd_alpha = 2 * H1.strain_data.roll_off / duration
H1_psd = H1_psd_data.psd(fftlength=duration, overlap=0, window=("tukey", psd_alpha), method="median")
L1_psd = L1_psd_data.psd(fftlength=duration, overlap=0, window=("tukey", psd_alpha), method="median")

H1.power_spectral_density = bilby.gw.detector.PowerSpectralDensity(
    frequency_array=H1_psd.frequencies.value, psd_array=H1_psd.value)
L1.power_spectral_density = bilby.gw.detector.PowerSpectralDensity(
    frequency_array=H1_psd.frequencies.value, psd_array=L1_psd.value)
H1.minimum_frequency = f_lower
L1.minimum_frequency = f_lower
H1.maximum_frequency = 1024
L1.maximum_frequency = 1024

In [ ]:
prior = bilby.core.prior.PriorDict()

prior['chirp_mass'] = #your code here
prior['mass_ratio'] = #your code here
prior["luminosity_distance"] = #your code here
prior["geocent_time"] = 2
prior["ra"] = 0
prior["dec"] = 0
prior["psi"] = 0
prior["theta_jn"] = 0
prior["phase"] = 0
prior["chi_1"] = 0
prior["chi_2"] = 0
prior["phase"] = 0

In [ ]:
# First, put our "data" created above into a list of interferometers (the order is arbitrary)
interferometers = [H1, L1]

# Next create a dictionary of arguments which we pass into the LALSimulation waveform - we specify the waveform approximant here
waveform_arguments = dict(
    waveform_approximant='IMRPhenomD', reference_frequency=50., catch_waveform_errors=True, minimum_frequency=f_lower)

# Next, create a waveform_generator object. This wraps up some of the jobs of converting between parameters etc
waveform_generator = bilby.gw.WaveformGenerator(
    frequency_domain_source_model=bilby.gw.source.lal_binary_black_hole,
    waveform_arguments=waveform_arguments,
    parameter_conversion=bilby.gw.conversion.convert_to_lal_binary_black_hole_parameters,
    sampling_frequency=sample_rate)

# Finally, create our likelihood, passing in what is needed to get going
likelihood = bilby.gw.likelihood.GravitationalWaveTransient(
    interferometers, waveform_generator, priors=prior, distance_marginalization=True)

In [ ]:
#Use this cell to run the sampler quickly for testing (it will still take a couple of minutes)
#also if your computer has more than 4 cores you can increase npool

result_short = bilby.run_sampler(
    likelihood, prior, sampler='dynesty', outdir='test_run', label="small",
    conversion_function=bilby.gw.conversion.generate_all_bbh_parameters,
    nlive=500, dlogz=3, npool=4
)

In [ ]:
#To get more accurate results, increase nlive to 1000 or more, and reduce dlogz to 0.1.
result_short = bilby.run_sampler(
    likelihood, prior, sampler='dynesty', outdir='full_run', label="full",
    conversion_function=bilby.gw.conversion.generate_all_bbh_parameters,
    nlive=1000, dlogz=0.1, npool=4
)

### Q4: Now you are going to investigate the relationship between chirp mass and sky localisation area. 

#### Q4a: Assuming all other parameters are fixed (including SNR), how would you expect the sky localization area to change if the chirp mass was increased or decreased? Give physical arguments. (Hint: consider the dependence of the GW localization on frequencies. Reference: https://journals.aps.org/prd/abstract/10.1103/PhysRevD.81.082001).

#### Q4b: Generate two signals with different chirp mass values but all other values the same. The chirp mass values should be significantly different e.g. a factor of 6 or more apart. Use your code from Q1 and Q2 to measure the SNR of each signal. Use this to estimate the luminosity distance you need to give each signal such that they both have the same SNR (you can assume all other parameters are fixed). You should aim for a relatively low SNR (e.g. less than 7) so that the sky localization areas are large enough to see a difference.


In [ ]:
#Your code here

#### Q4c: Pick a sky location, then use Bilby to estimate the sky location of each signal from Q4b. Plot the skymap for each signal using the code below. How does your result compare to your expectations from the physical arguments you gave above? 

Note: you should check in the output from running the initialise_bilby function that the Optimal SNRs for each signal are close to the same value. If not, tweak your luminosity distances until they are close. (eg if for signal 1, your H1 SNR is 6 and your L1 SNR is 7, signal 2 should have an H1 SNR of ~5.9-6.1 and an L1 SNR of ~6.9-7.1.).


In [ ]:
duration = 32.0
sampling_frequency = 4096.0
minimum_frequency = 20

mergertime = 1126259642.413

psd_duration = 1024
roll_off = 0.4

waveform_arguments = dict(
    waveform_approximant="IMRPhenomPv2",
    reference_frequency=50.0,
    minimum_frequency=minimum_frequency,
)

waveform_generator = bilby.gw.WaveformGenerator(
    duration=duration,
    sampling_frequency=sampling_frequency,
    frequency_domain_source_model=bilby.gw.source.lal_binary_black_hole,
    parameter_conversion=bilby.gw.conversion.convert_to_lal_binary_black_hole_parameters,
    waveform_arguments=waveform_arguments,
)

In [ ]:

def initialise_bilby(mass1, mass2, distance, right_ascension, declination):
	injection_parameters = dict(
		mass_1=mass1,
		mass_2=mass2,
		a_1=0,
		a_2=0,
		tilt_1=0,
		tilt_2=0,
		phi_12=0,
		phi_jl=0,
		luminosity_distance=distance,
		theta_jn=0,
		psi=0,
		phase=0,
		geocent_time=mergertime,
		ra=right_ascension,
		dec=declination,
	)


	ifos = bilby.gw.detector.InterferometerList(["H1", "L1"])
	ifos.set_strain_data_from_power_spectral_densities(
		sampling_frequency=sampling_frequency,
		duration=duration,
		start_time=mergertime - 20,
	)
	ifos.inject_signal(
		waveform_generator=waveform_generator, parameters=injection_parameters
	)

	priors = bilby.gw.prior.BBHPriorDict()

	del(priors['chirp_mass'])
	del(priors['mass_ratio'])
	#we don't modify the ra and dec distributions as those are set correctly by BBHPriorDict
	priors['mass_1'] = mass1
	priors['mass_2'] = mass2
	priors["luminosity_distance"] = distance
	priors["geocent_time"] = mergertime
	priors["psi"] = 0
	priors["theta_jn"] = 0
	priors["phase"] = 0
	priors["a_1"] = 0
	priors["a_2"] = 0
	priors["tilt_1"] = 0
	priors["tilt_2"] = 0
	priors["phi_12"] = 0
	priors["phi_jl"] = 0

	likelihood = bilby.gw.GravitationalWaveTransient(
		interferometers=ifos, waveform_generator=waveform_generator,priors=priors
	)

	print(priors)
	return likelihood, priors, injection_parameters


In [ ]:
#small test run. For the real run increase npoints to 1000 or more and reduce dlogz to 0.1
#note that the full run could take several hours depending on your injection parameters and computer.
likelihood, priors, injection_parameters = initialise_bilby()
result = bilby.run_sampler(
    likelihood=likelihood,
	priors=priors,
    sampler="dynesty",
    npoints=100,
    injection_parameters=injection_parameters, dlogz=3,
    outdir='short_test', label="test", result_class=bilby.gw.result.CBCResult, npool=4
)


In [ ]:
#use this to plot your skymap. Note it could take a few minutes to run.
result.plot_skymap(maxpts=1000)

#### Q4c: Compare your localisation areas with the GWTC-3 catalogue. You should find that your areas are significantly smaller compared to signals with similar SNR. Why is this? To investigate, choose some of the prior values to be uniform over a range rather than fixed, and rerun the analysis. Note that if you choose to vary the distance, phase or time, you should specify distance_marginalization=True, phase_marginalisation = True, etc, in the likelihood function.
